# E11 — Differentiable simulation vs PPO (GPU runs)

Runs the baseline comparison of `EXPERIMENTS.md` (E11) on a Colab GPU.
Every output (logs, checkpoints, figure) is written to **Google Drive**, so a
disconnected runtime loses at most the run in progress.

**Before starting:** `Runtime → Change runtime type → GPU`.

Run the cells in order. Each training cell is one (method, seed) pair: if the
runtime disconnects, reconnect, rerun the setup cells (1–4) and continue from
the first cell whose checkpoint is missing on Drive.

## 1. GPU check

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Google Drive (persistent outputs)

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/watercrea_E11"
!mkdir -p {OUT}/runs {OUT}/diffsim {OUT}/ppo

## 3. Code

In [ ]:
%cd /content
!rm -rf watercrea && git clone -q https://github.com/mrlrns/parallel-neuroevolution-engine.git watercrea
%cd /content/watercrea
!git log --oneline -1
!pip install -q pyyaml matplotlib

## 4. Environment check (must print ✅)

In [ ]:
!python -m baselines.check_env --chemin-champion champion_raffine/reference.pt

## 5. Timing test

5 episodes of each method at full size, into a scratch folder. Multiply by
60 (300 episodes / 5) to estimate one full run, then by 6 for the whole grid.
If it does not fit your Colab budget, lower `NB` below (same value for both
methods, so the comparison stays at equal budget).

In [ ]:
CHAMP = "champion_raffine/reference.pt"
!time python train2.py --chemin-champion {CHAMP} --poids-aleatoires --nb-episodes 5 --dossier-runs /tmp/t --dossier-champion-raffine /tmp/t
!time python -m baselines.ppo --chemin-champion {CHAMP} --poids-aleatoires --ppo-nb-iterations 5 --dossier-runs /tmp/t --dossier-baselines /tmp/t

## 6. Full runs — one cell per (method, seed)

`NB` = episodes for train2 = iterations for PPO → identical `sim_steps` budget.

In [ ]:
NB = 300


def diffsim(seed):
    !python train2.py --chemin-champion {CHAMP} --poids-aleatoires --seed {seed} --nb-episodes {NB} --dossier-runs {OUT}/runs --dossier-champion-raffine {OUT}/diffsim


def ppo(seed):
    !python -m baselines.ppo --chemin-champion {CHAMP} --poids-aleatoires --seed {seed} --ppo-nb-iterations {NB} --dossier-runs {OUT}/runs --dossier-baselines {OUT}/ppo

In [ ]:
diffsim(0)

In [ ]:
ppo(0)

In [ ]:
diffsim(1)

In [ ]:
ppo(1)

In [ ]:
diffsim(2)

In [ ]:
ppo(2)

## 7. Learning curves vs simulation budget

In [ ]:
from IPython.display import Image

!python -m baselines.compare --diffsim "{OUT}/runs/train_phase2_seed*.jsonl" --ppo "{OUT}/runs/ppo_seed*.jsonl" --sortie {OUT}/baseline_ppo.png
Image(f"{OUT}/baseline_ppo.png")

## 8. Final evaluation — same judge for every checkpoint

20 seeds × 1000 frames per checkpoint (`evaluate_seeds.py`). Copy the
`top 5` and the per-seed displacements into E11.

In [ ]:
import glob

checkpoints = sorted(glob.glob(f"{OUT}/diffsim/FINAL_seed*.pt")) + sorted(glob.glob(f"{OUT}/ppo/ppo_seed*_best.pt"))
for ckpt in checkpoints:
    print("=" * 80, "\n", ckpt)
    !python evaluate_seeds.py --chemin-champion "{ckpt}" --nb-seeds 20 | tail -8

## 9. Bring the results back

Download `baseline_ppo.png` from Drive into `docs/` in the repository, fill the
E11 table and write the conclusion yourself.